# Spatial operations

In this notebook we will do spatial operations using DuckDB and Python.

<a target="_blank" href="https://colab.research.google.com/github/kraina-ai/srai-tutorial/blob/dawts2026/02_spatial_operations.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

In [ ]:
## Uncomment the line below when running on Google Colab
# %pip install geopandas pooch folium branca mapclassify h3 shapely quackosm contextily

## Spatial joins and operations

Here we will join two spatial datasets together using different predicates with DuckDB and GeoPandas.

### Intersects

We will download Warsaw boundaries, green spaces and calculate what district is the greenest.

In [ ]:
import geopandas as gpd
import overturemaestro as om
import quackosm as qosm
import duckdb

duckdb.install_extension("spatial")
duckdb.load_extension("spatial")

warsaw_boundary = qosm.geocode_to_geometry("Warsaw")
district_boundaries_url = "https://github.com/andilabs/warszawa-dzielnice-geojson/raw/refs/heads/master/warszawa-dzielnice.geojson"

display(warsaw_boundary)

#### DuckDB - SQL

In [ ]:
greenery_filter = {
    "landuse": ["grass", "greenery", "recreation_ground", "forest"],
    "leisure": ["park"],
}

In [ ]:
green_regions_parquet = qosm.convert_geometry_to_parquet(
    geometry_filter=warsaw_boundary, tags_filter=greenery_filter
)
green_regions_parquet

In [ ]:
duckdb.sql(
    f"""
    SELECT
        *, ST_Area(geom)
    FROM ST_Read('{district_boundaries_url}')
    """
).show(max_width=120)

In [ ]:
duckdb.sql(
    f"""
    SELECT
        *
    FROM read_parquet('{green_regions_parquet}')
    """
).show(max_width=120)

Join the data spatially with ST_Intersects (geometry intersects another geometry)

In [ ]:
duckdb.sql(
    f"""
    WITH districts AS (
        SELECT
            name, ST_SetCRS(geom, 'EPSG:4326') geometry
        FROM ST_Read('{district_boundaries_url}')
        WHERE name != 'Warszawa'
    ),
    greenery AS (
        SELECT
            feature_id, ST_SetCRS(geometry, 'EPSG:4326') geometry
        FROM read_parquet('{green_regions_parquet}')
    )
    SELECT *
    FROM districts
    JOIN greenery
    ON ST_Intersects(districts.geometry, greenery.geometry)
    """
).show(max_width=250)

Calculate intersections and areas

In [ ]:
duckdb.sql(
    f"""
    SELECT
            feature_id, ST_GeometryType(geometry), ST_SetCRS(geometry, 'EPSG:4326') geometry
        FROM read_parquet('{green_regions_parquet}')
        WHERE ST_GeometryType(geometry) != 'POINT'
    """
).show(max_width=120)

In [ ]:
duckdb.sql(
    f"""
    WITH districts AS (
        SELECT
            name, ST_SetCRS(geom, 'EPSG:4326') geometry
        FROM ST_Read('{district_boundaries_url}')
        WHERE name != 'Warszawa'
    ),
    greenery AS (
        SELECT
            feature_id, ST_SetCRS(geometry, 'EPSG:4326') geometry
        FROM read_parquet('{green_regions_parquet}')
    ),
    greenery_in_districts AS (
        SELECT
            districts.name,
            SUM(
                ST_Area(
                    ST_Intersection(
                        districts.geometry,
                        greenery.geometry
                    )
                )
            ) intersection_area
        FROM districts
        JOIN greenery
        ON ST_Intersects(districts.geometry, greenery.geometry)
        GROUP BY 1
    )
    SELECT
        greenery_in_districts.name,
        ROUND(100 * intersection_area / ST_Area(districts.geometry), 2) as greenery_area_pct
    FROM greenery_in_districts
    JOIN districts
    USING (name)
    ORDER BY 2 DESC
    """
)

#### GeoPandas

In [ ]:
district_boundaries = gpd.read_file(district_boundaries_url) #.to_crs(2180)
district_boundaries = district_boundaries[district_boundaries["name"] != "Warszawa"]
district_boundaries['area'] = district_boundaries.area
district_boundaries

In [ ]:
green_regions_gdf = qosm.convert_geometry_to_geodataframe(
    geometry_filter=warsaw_boundary, tags_filter=greenery_filter
)
green_regions_gdf = green_regions_gdf[green_regions_gdf.geom_type != "Point"]
green_regions_gdf

In [ ]:
greenery_in_disticts_joined = district_boundaries.overlay(
    green_regions_gdf, how="intersection"
)
greenery_in_disticts_joined["greenery_area"] = greenery_in_disticts_joined.area
greenery_in_disticts_joined

In [ ]:
greenery_in_disticts = greenery_in_disticts_joined.groupby("name").agg(
    district_area=("area", "first"), greenery_area=("greenery_area", "sum")
)
greenery_in_disticts["greenery_area_pct"] = (
    100 * greenery_in_disticts["greenery_area"] / greenery_in_disticts["district_area"]
).round(2)
greenery_in_disticts.sort_values("greenery_area_pct", ascending=False)

In [ ]:
district_boundaries.join(greenery_in_disticts, on="name").explore(
    "greenery_area_pct",
    tiles="CartoDB Positron",
    cmap="Greens",
    style_kwds=dict(color="black", weight=0.5),
    tooltip=["name", "greenery_area_pct"],
)

### DWithin (Distance Within)

Here we will download Warsaw buildings, cafes and calculate spaces where there is a low coverage of cafes nearby.

To operate in meters, we have to change the CRS from current latitude / longitude into EPSG:2180 - Polish projected coordinate reference system.

In [ ]:
bakery_filter = {"shop": "bakery"}
buildings_filter = {"building": True}
BAKERY_CHECK_RADIUS_METERS = 2000

Prepare required inputs

In [ ]:
buildings_parquet = qosm.convert_geometry_to_parquet(
    geometry_filter=warsaw_boundary, tags_filter=buildings_filter
)
buildings_gdf = qosm.convert_geometry_to_geodataframe(
    geometry_filter=warsaw_boundary, tags_filter=buildings_filter
)

bakeries_parquet = qosm.convert_geometry_to_parquet(
    geometry_filter=warsaw_boundary, tags_filter=bakery_filter
)
bakeries_gdf = qosm.convert_geometry_to_geodataframe(
    geometry_filter=warsaw_boundary, tags_filter=bakery_filter
)

#### Duckdb - SQL

First let's see how ST_Dwithin works

In [ ]:
duckdb.sql(
    f"""
    WITH buildings AS (
        SELECT
            feature_id as building_id,
            ST_Transform(
                ST_Centroid(geometry),
                'EPSG:2180', 
                always_xy := true
            ) AS geometry
        FROM '{buildings_parquet}'
    ),
    bakeries AS (
        SELECT
            feature_id as bakery_id,
            ST_Transform(
                ST_Centroid(geometry),
                'EPSG:2180', 
                always_xy := true
            ) AS geometry
        FROM '{bakeries_parquet}'
    )
    SELECT *
    FROM buildings
    JOIN bakeries
    ON ST_DWithin(buildings.geometry, bakeries.geometry, {BAKERY_CHECK_RADIUS_METERS})
    """
).show(max_width=120)

In [ ]:
bakeries_around_buildings = duckdb.sql(
    f"""
    WITH buildings AS (
        SELECT
            feature_id as building_id,
            ST_Transform(
                ST_Centroid(geometry),
                'EPSG:2180', 
                always_xy := true
            ) AS geometry
        FROM '{buildings_parquet}'
    ),
    bakeries AS (
        SELECT
            feature_id as bakery_id,
            ST_Transform(
                ST_Centroid(geometry),
                'EPSG:2180', 
                always_xy := true
            ) AS geometry
        FROM '{bakeries_parquet}'
    ),
    bakeries_around_buildings AS (
        SELECT buildings.building_id, COUNT(bakeries.bakery_id) as bakeries
        FROM buildings
        JOIN bakeries
        ON ST_DWithin(buildings.geometry, bakeries.geometry, {BAKERY_CHECK_RADIUS_METERS})
        GROUP BY 1
    )
    SELECT building_id, COALESCE(bakeries, 0) as bakeries, ST_AsWKT(geometry) wkt
    FROM buildings
    LEFT JOIN bakeries_around_buildings
    USING (building_id)
    ORDER BY 2 DESC
    """
)
bakeries_around_buildings.limit(10).show(max_width=120)

In [ ]:
bakeries_around_buildings_gdf = bakeries_around_buildings.to_df()
bakeries_around_buildings_gdf = gpd.GeoDataFrame(
    bakeries_around_buildings_gdf,
    geometry=gpd.GeoSeries.from_wkt(bakeries_around_buildings_gdf["wkt"], crs=2180),
)
bakeries_around_buildings_gdf

In [ ]:
bakeries_around_buildings_gdf.plot("bakeries")

#### GeoPandas

In [ ]:
buildings_gdf = (
    buildings_gdf.to_crs(2180)
    .reset_index()
    .rename(columns={"feature_id": "building_id"})
)
bakeries_gdf = (
    bakeries_gdf.to_crs(2180).reset_index().rename(columns={"feature_id": "bakery_id"})
)

In [ ]:
buildings_with_bakeries = buildings_gdf.sjoin(
    bakeries_gdf, how="left", predicate="dwithin", distance=BAKERY_CHECK_RADIUS_METERS
).reset_index()
buildings_with_bakeries

In [ ]:
bakeries_around_buildings_gdf_v2 = gpd.GeoDataFrame(
    buildings_with_bakeries.groupby("building_id")
    .agg(bakeries=("bakery_id", "nunique"))
    .sort_values("bakeries", ascending=False)
    .reset_index()
    .merge(buildings_gdf, on="building_id")
)
bakeries_around_buildings_gdf_v2.geometry = bakeries_around_buildings_gdf_v2.centroid
bakeries_around_buildings_gdf_v2

In [ ]:
bakeries_around_buildings_gdf_v2.plot("bakeries")

### Buffering and Intersections

Here we will calculate best places for the family by searching places close to schools, public transport stops and supermarkets.

We will intersect together best locations and show it on a map.

In [ ]:
public_transport_filter = {"public_transport": "stop_position"}
# shop_filter = {"shop": ['supermarket', 'convenience']}
shop_filter = {"shop": 'supermarket'}
schools_filter = {"amenity": "school"}

PUBLIC_TRANSPORT_BUFFER_METERS = 300
SHOP_BUFFER_METERS = 400
SCHOOL_BUFFER_METERS = 500

In [ ]:
public_transport_gdf = qosm.convert_geometry_to_geodataframe(
    geometry_filter=warsaw_boundary, tags_filter=public_transport_filter
).to_crs(2180)
shops_gdf = qosm.convert_geometry_to_geodataframe(
    geometry_filter=warsaw_boundary, tags_filter=shop_filter
).to_crs(2180)
schools_gdf = qosm.convert_geometry_to_geodataframe(
    geometry_filter=warsaw_boundary, tags_filter=schools_filter
).to_crs(2180)

In [ ]:
public_transport_gdf.geometry = public_transport_gdf.centroid.buffer(PUBLIC_TRANSPORT_BUFFER_METERS)
shops_gdf.geometry = shops_gdf.centroid.buffer(SHOP_BUFFER_METERS)
schools_gdf.geometry = schools_gdf.centroid.buffer(SCHOOL_BUFFER_METERS)

public_transport_gdf

In [ ]:
import matplotlib.pyplot as plt
import contextily as cx

fig, ax = plt.subplots(figsize=(10, 10))

public_transport_gdf.boundary.to_crs(4326).plot(ax=ax, color="orange", lw=0.5)
schools_gdf.boundary.to_crs(4326).plot(ax=ax, color="limegreen", lw=0.5)
shops_gdf.boundary.to_crs(4326).plot(ax=ax, color="royalblue", lw=0.5)

cx.add_basemap(ax=ax, source=cx.providers.CartoDB.VoyagerNoLabels, crs=4326, zoom=12)

plt.show()

In [ ]:
public_transport_union = public_transport_gdf.union_all()
public_transport_union_gdf = gpd.GeoDataFrame(geometry=[public_transport_union], crs=2180)
public_transport_union

In [ ]:
shops_union = shops_gdf.union_all()
shops_union_gdf = gpd.GeoDataFrame(geometry=[shops_union], crs=2180)
shops_union

In [ ]:
schools_union = schools_gdf.union_all()
schools_union_gdf = gpd.GeoDataFrame(geometry=[schools_union], crs=2180)
schools_union

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

public_transport_union_gdf.to_crs(4326).plot(
    ax=ax, color=(0, 0, 0, 0), hatch="///", edgecolor="orange", lw=0.5
)
shops_union_gdf.to_crs(4326).plot(
    ax=ax, color=(0, 0, 0, 0), hatch="|||", edgecolor="royalblue", lw=0.5
)
schools_union_gdf.to_crs(4326).plot(
    ax=ax, color=(0, 0, 0, 0), hatch="\\\\\\", edgecolor="limegreen", lw=0.5
)

cx.add_basemap(ax=ax, source=cx.providers.CartoDB.VoyagerNoLabels, crs=4326, zoom=12)

plt.show()

In [ ]:
best_locations_geometry = public_transport_union.intersection(shops_union).intersection(
    schools_union
)
best_locations_geometry_gdf = gpd.GeoDataFrame(
    geometry=[best_locations_geometry], crs=2180
)
best_locations_geometry

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))


best_locations_geometry_gdf.to_crs(4326).plot(
    ax=ax, color=(0, 0, 0, 0), hatch="///", edgecolor="orange", lw=0.5
)
best_locations_geometry_gdf.boundary.to_crs(4326).plot(
    ax=ax, color="black", lw=0.5
)
cx.add_basemap(ax=ax, source=cx.providers.CartoDB.VoyagerNoLabels, crs=4326, zoom=12)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

best_locations_geometry_gdf.to_crs(4326).plot(
    ax=ax, color=(0, 0, 0, 0), hatch="///", edgecolor="orange", lw=0.5
)
best_locations_geometry_gdf.boundary.to_crs(4326).plot(
    ax=ax, color="black", lw=0.5
)

ax.set_xlim(21.014614, 21.045942)
ax.set_ylim(52.148975, 52.167615)

cx.add_basemap(ax=ax, source=cx.providers.CartoDB.VoyagerNoLabels, crs=4326, zoom=16)

plt.show()

## Transactions data analysis - RCN dataset for Warsaw

We will use public data from deweloperuch github repository and aggregate it into a heatmap using H3 cells.

In [ ]:
from pooch import retrieve, Unzip

In [ ]:
RCN_WARSAW_URL = 'https://github.com/deweloperuch/rejestr-cen-nieruchomosci/raw/refs/heads/main/data/warszawa/warszawa_rcn.zip'

rcn_file_path = retrieve(RCN_WARSAW_URL, path='files/rcn', known_hash=None, processor=Unzip())[0]

In [ ]:
import geopandas as gpd

gpd.list_layers(rcn_file_path)

In [ ]:
flats = gpd.read_file(rcn_file_path, layer="RCN_Lokal").rename(
    columns={"powUzytkowaLokalu": "area_m2", "cenaLokaluBrutto": "price"}
)
flats

In [ ]:
flats.plot(markersize=5)

In [ ]:
flats["price_per_m2"] = (flats["price"] / flats["area_m2"]).round(2)
flats

In [ ]:
flats["price_per_m2"].describe(
    [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).round(2)

In [ ]:
flats_filtered = flats[
    flats["price_per_m2"].between(
        flats["price_per_m2"].quantile(0.01), flats["price_per_m2"].quantile(0.99)
    )
].copy()
flats_filtered["price_per_m2"].hist()

In [ ]:
flats_filtered.plot("price_per_m2", legend=True)

In [ ]:
import h3


def point_to_h3(point):
    lon = point.x
    lat = point.y
    return h3.latlng_to_cell(lat, lon, 8)


flats_filtered["h3"] = flats_filtered.geometry.to_crs(4326).apply(point_to_h3)
flats_filtered

In [ ]:
aggregated_flats = (
    flats_filtered.groupby("h3")
    .agg(
        price_per_m2=("price_per_m2", "median"),
        total_transactions=("gml_id", "nunique"),
    )
    .round(2)
    .reset_index()
)
aggregated_flats

In [ ]:
from shapely import Polygon


def h3_to_geometry(h3_id):
    coords = h3.cell_to_boundary(h3_id)
    flipped = tuple(coord[::-1] for coord in coords)
    return Polygon(flipped)


aggregated_flats = gpd.GeoDataFrame(
    aggregated_flats, geometry=aggregated_flats["h3"].apply(h3_to_geometry), crs=4326
)
aggregated_flats

In [ ]:
aggregated_flats.explore(
    "price_per_m2",
    cmap="RdYlGn_r",
    tiles="CartoDB Voyager",
    style_kwds=dict(color="black", weight=0.5),
)